# Questão 2 – Forma de Hessenberg

## Sumário

- [Item a)](#item-a) - Aplicando refletores: função `apply_reflector(v, beta, b)`
- [Item b)](#item-b) - Testes de corretude e complexidade $O(n)$
- [Item c)](#item-c) - Generalização para matrizes: `apply_reflector(v, beta, A)`
- [Item d)](#item-d) - Função `rev_apply_reflector(v, beta, A)`
- [Item e)](#item-e) - Redução à forma de Hessenberg: `to_hessenberg(A)`
- [Item f)](#item-f) - Verificação de corretude

---

In [15]:
using LinearAlgebra, Plots

## <a id="item-a"></a> Item a)

**Enunciado:** Escreva uma função `apply_reflector(v, beta, b)` que calcula $Q_v b$, onde $Q_v = I - \beta v v^*$ é o refletor de Householder dado por $v$ e $\beta$.

**Solução:**

---

In [7]:
# função foi adaptada devido ao item b
function apply_reflector(v::AbstractVector{T}, beta::T, b::AbstractVector{T}) where {T<:AbstractFloat}
    size_v = length(v)
    size_b = length(b)

    if size_v > size_b # Verificando se é possível aplicar o refletor
        error("v não pode ser maior que b")
    end

    # Modificando apenas onde é necessário
    b_sub = @view b[size_b - size_v + 1:end] # Sem precisar fazer alocação extra 
    b_sub .-= (beta * dot(v, b_sub)) .* v 

    return b
end

apply_reflector (generic function with 1 method)

## <a id="item-b"></a> Item b)

**Enunciado:** Verifique que sua função está correta, aplicando em vetores $x$ de mesma dimensão que $v$, e depois para vetores de dimensões maiores do que $v$ (adaptando, se necessário, sua função para funcionar neste caso). Certifique-se que sua função tem complexidade $O(n)$, onde $n$ é a dimensão do vetor de entrada.

**Solução:**

### Checagem de custo
 
Sejam $n = \dim(b)$ e $m = \dim(v)$, com $n \geq m$ (caso contrário a aplicação não é definida).
 
- **Obter os tamanhos** (`length(v)`, `length(b)`) e **verificar** $m \leq n$: custo $O(1)$ cada — acessar o tamanho de um vetor em Julia é uma operação de tempo constante.
- **`@view b[end-m+1:end]`**: cria uma *view* da parte inferior de $b$ sem alocar memória adicional — custo $O(1)$.
As operações principais são então, em ordem:
 
1. **Produto interno** $v^\top b_{\text{sub}}$: $2m - 1$ operações (multiplicações e adições) — custo $O(m)$.
2. **Escalar** $\beta \cdot (v^\top b_{\text{sub}})$: uma multiplicação — custo $O(1)$.
3. **Escalar o vetor** $(\beta \cdot v^\top b_{\text{sub}}) \cdot v$ via `.*`: $m$ multiplicações — custo $O(m)$.  
   Vale destacar que o broadcasting `.*` de Julia opera elemento a elemento sem construir vetores intermediários, o que é tanto eficiente em memória quanto em tempo pois não gasta operações com essas alocações.
4. **Subtração** `b_sub .-= ...`: $m$ subtrações in-place — custo $O(m)$.
O custo total é $O(1) + O(1) + O(m) + O(1) + O(m) + O(m) = O(m)$. Como $m \leq n$, temos $O(m) \subseteq O(n)$, portanto a complexidade da função é $O(n)$ como requerido. $\square$


---

In [8]:
# OBSERVAÇÃO: Cópia da função reflector da questão 1 para conseguir usa-lá nesse notebook

function reflector(x::AbstractVector{T}) where {T<:AbstractFloat}
    norm2y = dot(@view(x[2:end]), @view(x[2:end])) # View permite fazer sem alocação extra
    normx  = sqrt(norm2y + x[1]^2) # Norma de x
    v      = copy(x)
 
    if x[1] > zero(T)
        v[1] = -norm2y / (normx + x[1])
        b = (normx + x[1]) / (normx * norm2y)
    else
        v[1] = x[1] - normx
        b = one(T) / (normx * (normx - x[1]))
    end
    
    return v, b
end

reflector (generic function with 1 method)

In [9]:
# Vetor de teste (Mesmo vetor que usamos na questão 1)
x0 = [10000.0, 6400.0, 4900.0, 8100.0, 2500.0, 3600.0, 4000.0, 1600.0]
v, beta = reflector(x0)

println("dim(b) = dim(v)")
b = copy(x0)
result = apply_reflector(v, beta, b)
e1 = [one(Float64); zeros(Float64, length(x0) - 1)]
println("apply_reflector(v, beta, x0) = ", result)
println("||x0|| * e1 = ", norm(x0) * e1)
println("Diff relativa: ", norm(result - norm(x0) * e1) / norm(norm(x0) * e1))

println()

println("dim(b) > dim(v)")
b = [1.0, 2.0, 3.0, 4.0, x0...]  # primeiros 4 elementos não devem mudar
result = apply_reflector(v, beta, b)
println("Primeiros elementos (não devem mudar): ", result[1:4])
println("Últimos elementos (||x0||*e1): ", result[5:end])
println("Dif relativa na parte inferior: ", norm(result[5:end] - norm(x0) * e1) / norm(norm(x0) * e1))

dim(b) = dim(v)
apply_reflector(v, beta, x0) = [16381.391882254695, -1.8189894035458565e-12, -9.094947017729282e-13, -1.8189894035458565e-12, -4.547473508864641e-13, -9.094947017729282e-13, -9.094947017729282e-13, -4.547473508864641e-13]
||x0|| * e1 = [16381.391882254695, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Diff relativa: 1.8827744399455417e-16

dim(b) > dim(v)
Primeiros elementos (não devem mudar): [1.0, 2.0, 3.0, 4.0]
Últimos elementos (||x0||*e1): [16381.391882254695, -1.8189894035458565e-12, -9.094947017729282e-13, -1.8189894035458565e-12, -4.547473508864641e-13, -9.094947017729282e-13, -9.094947017729282e-13, -4.547473508864641e-13]
Dif relativa na parte inferior: 1.8827744399455417e-16


## <a id="item-c"></a> Item c)

**Enunciado:** Generalize sua função para `apply_reflector(v, beta, A)` que calcula $Q_v A$ para uma matriz $A$ (com no mínimo o mesmo número de linhas do que $v$).

**Solução:**


Sabemos que, escrevendo $A = [A_1, A_2, \ldots, A_m]$ em termos de suas colunas, a multiplicação por uma matriz $B$ satisfaz:

$$BA = [BA_1, BA_2, \ldots, BA_m].$$

Portanto, aplicar o refletor $Q_v$ à matriz $A$ é equivalente a aplicar $Q_v$ a cada coluna de $A$ individualmente:

$$Q_v A = [Q_v A_1, Q_v A_2, \ldots, Q_v A_m].$$

Isso é computacionalmente vantajoso (Como vimos em aula): em vez de montar explicitamente a matriz $Q_v = I - \beta vv^\top$ (custo $O(n^2)$ em memória e operações), operamos diretamente com $v$ e $\beta$, aplicando a reflexão coluna a coluna via `apply_reflector`. Isso reduz o custo para $O(nm)$ e evita armazenar $Q_v$.

In [10]:
function apply_reflector(v::AbstractVector{T}, beta::T, A::AbstractMatrix{T}) where {T<:AbstractFloat}
    for j in axes(A, 2)
        col = @view A[:, j]
        apply_reflector(v, beta, col)
    end
    return A
end

apply_reflector (generic function with 2 methods)


---

## <a id="item-d"></a> Item d)

**Enunciado:** Escreva uma função `rev_apply_reflector(v, beta, A)` que calcula $A Q_v$ (com $A$ de dimensões compatíveis).

**Solução:**

Sabemos que aplicar uma transformação pela **esquerda** ($Q_v A$) opera nas **linhas** de $A$, enquanto aplicar pela **direita** ($A Q_v$) opera nas **colunas** de $A$.

Como $Q_v = I - \beta v v^\top$, temos:

$$AQ_v = A(I - \beta v v^\top) = A - \beta (Av) v^\top$$

Comparando com o caso anterior:

$$Q_v A = A - \beta v (v^\top A)$$

Ou seja, em $Q_v A$ o vetor $v$ aparece à **esquerda** (opera nas linhas), e em $AQ_v$ o vetor $v^\top$ aparece à **direita** (opera nas colunas). Analogamente ao item c, podemos escrever $A = [a_1, \ldots, a_m]^\top$ em termos de suas **linhas** e aplicar a reflexão linha a linha.

> **Observação:** Quando $A$ é simétrica, $AQ_v = Q_v A$ — fato que será explorado no item f para verificar a corretude da implementação.

### Implementação

In [11]:
function rev_apply_reflector(v::AbstractVector{T}, beta::T, A::AbstractMatrix{T}) where {T<:AbstractFloat}
    for i in axes(A, 1)
        row = @view A[i, :]
        apply_reflector(v, beta, row)
    end
    return A
end

rev_apply_reflector (generic function with 1 method)


---

## <a id="item-e"></a> Item e)

**Enunciado:** Escreva uma função `to_hessenberg(A)` que calcula a forma de Hessenberg de uma matriz $A$ usando refletores de Householder. A função deve retornar uma lista de refletores $(v_i, \beta_i)$, a matriz $H$ tal que $A = Q H Q^*$, e, opcionalmente, $Q$, que é a matriz ortogonal dada pelo produto dos refletores.

**Solução:**

In [19]:
function to_hessenberg(A::Matrix{T}; compute_Q=false) where {T<:AbstractFloat}
    refletores = Vector{Tuple{Vector{T},T}}()
    H          = copy(A)
    n          = size(A, 1)

    for j in 1:n-2
        x = H[j+1:n, j]
        v, beta = reflector(x)

        push!(refletores, (copy(v), beta))

        H_sub = @view H[j+1:n, j:n]
        apply_reflector(v, beta, H_sub)

        H_sub2 = @view H[1:n, j+1:n]
        rev_apply_reflector(v, beta, H_sub2)
    end

    if compute_Q
        Q = Matrix{T}(I, n, n)
        for (j, (v, beta)) in enumerate(refletores)
            Q_sub = @view Q[j+1:n, :]
            apply_reflector(v, beta, Q_sub)
        end
        return refletores, H, Matrix(Q')
    end

    return refletores, H
end

to_hessenberg (generic function with 2 methods)


---

## <a id="item-f"></a> Item f)

**Enunciado:** Verifique que sua função de fato está correta, calculando $\|A - Q H Q^*\|$ e $\|Q^* Q - I\|$ para matrizes simétricas e não simétricas, e de tamanhos $2$, $10$ e $100$.

**Solução:**



In [ ]:
function test_hessenberg(n::Int, T::Type{<:AbstractFloat}; symmetric=false)
    # Gerando matrizes aleatórias
    M = T.(randn(n, n))
    A = symmetric ? (M + M') / 2 : M # Forma simétrica de M

    refletores, H = to_hessenberg(A)

    # Construindo Q como produto dos refletores
    Q = Matrix{T}(I, n, n)
    for (v, beta) in refletores
        apply_reflector(v, beta, Q)
    end
    Q = Q'  # Q* = Q^T para ortogonais (Nos reais) OBS: Estou supondo reais porque na questão 1 testamos com Float

    err_hess = norm(A - Q * H * Q')
    err_orth = norm(Q' * Q - I)

    println("n=$n, simétrica=$symmetric, tipo=$T")
    println("  ||A - QHQ*|| = $err_hess")
    println("  ||Q*Q - I||  = $err_orth")
    println()
end

# Testes do enunciado
for n in [2, 10, 100]
    for sym in [false, true]
        test_hessenberg(n, Float64; symmetric=sym)
    end
end

n=2, simétrica=false, tipo=Float64
  ||A - QHQ*|| = 0.0
  ||Q*Q - I||  = 0.0

n=2, simétrica=true, tipo=Float64
  ||A - QHQ*|| = 0.0
  ||Q*Q - I||  = 0.0

n=10, simétrica=false, tipo=Float64
  ||A - QHQ*|| = 5.839069170483479e-15
  ||Q*Q - I||  = 1.3193555692410922e-15

n=10, simétrica=true, tipo=Float64
  ||A - QHQ*|| = 3.969847873902262e-15
  ||Q*Q - I||  = 1.5367408007019565e-15

n=100, simétrica=false, tipo=Float64
  ||A - QHQ*|| = 1.1000398765266328e-13
  ||Q*Q - I||  = 8.577425705056516e-15

n=100, simétrica=true, tipo=Float64
  ||A - QHQ*|| = 7.173682211367392e-14
  ||Q*Q - I||  = 8.503514091487391e-15

